# 19 Batch presets

This notebook gives you practical preset modes for running the pipeline.
It reuses the orchestration logic and exports preset-specific status/plan tables.


In [1]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.batch_presets import (
    build_preset_plan,
    build_preset_status,
    export_preset_tables,
    list_presets,
    summarize_preset,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


PROJECT_ROOT = C:\00_Developement\sch-file-organizer
OUTPUT_DIR = C:\00_Developement\sch-file-organizer\data\outputs


In [2]:
PRESET_NAME = 'review_only'

# Valid options:
# review_only
# canonicalize_unresolved
# ocr_rescue
# promote_to_execution
# apply_small_live_batch
# rollback_last_batch


In [3]:
presets_df = list_presets()
display(presets_df)


,name,description,include_ocr,include_apply,include_rollback,include_quality,goal,notes
0,review_only,Run the safe read-only core pipeline up to rev...,False,False,False,False,"Inspect current files, duplicates, junk, extra...",Best default for a new batch or after policy c...
1,canonicalize_unresolved,Run the quality branch to canonicalize unresol...,False,False,False,True,Improve deterministic canonical paths and name...,Use after review outputs and before any execut...
2,ocr_rescue,Run the OCR branch for scanned PDFs and image-...,True,False,False,True,Rescue weak/no-text files so content-derived f...,Use only on a small sandbox or a targeted subs...
3,promote_to_execution,Promote newly canonical-ready rows back into p...,True,False,False,True,Convert accepted deterministic results into pl...,Use after feedback reruns have improved unreso...
4,apply_small_live_batch,Prepare and execute a small approved batch wit...,True,True,False,True,"Run a tiny live batch only after dry-run, revi...",Always use a copied sandbox or a tightly bound...
5,rollback_last_batch,Rollback the most recent moved batch and then ...,True,True,True,True,Reverse the last applied batch and verify obse...,Only meaningful after a real moved batch exists.


In [4]:
summary_df = summarize_preset(OUTPUT_DIR, PRESET_NAME)
status_df = build_preset_status(OUTPUT_DIR, PRESET_NAME)
plan_df = build_preset_plan(OUTPUT_DIR, PRESET_NAME)

display(summary_df)
display(status_df)
display(plan_df)


,preset_name,description,goal,notes,done_count,pending_count,next_notebook
0,review_only,Run the safe read-only core pipeline up to rev...,"Inspect current files, duplicates, junk, extra...",Best default for a new batch or after policy c...,7,0,None


,preset_name,stage,branch,optional,notebook,description,output_patterns,latest_file,available
0,review_only,01_policy_check,core,False,01_policy_check.ipynb,Validate YAML policy and vocabularies.,,NaN,True
1,review_only,02_inventory,core,False,02_inventory.ipynb,Scan files and write inventory parquet/csv.,inventory_*.parquet,C:\00_Developement\sch-file-organizer\data\out...,True
2,review_only,03_rule_classification,core,False,03_rule_classification.ipynb,Apply deterministic rules to inventory.,rule_classification_*.parquet,C:\00_Developement\sch-file-organizer\data\out...,True
3,review_only,04_extract_text,core,False,04_extract_text.ipynb,Extract text previews from supported file types.,inventory_with_text_*.parquet,C:\00_Developement\sch-file-organizer\data\out...,True
4,review_only,05_review_outputs,core,False,05_review_outputs.ipynb,Merge outputs into one review frame.,review_snapshot_latest.parquet,C:\00_Developement\sch-file-organizer\data\out...,True
5,review_only,06_planner,core,False,06_planner.ipynb,Build conservative dry-run plan.,plan_dry_run_*.parquet,C:\00_Developement\sch-file-organizer\data\out...,True
6,review_only,07_execution_manifest,core,False,07_execution_manifest.ipynb,"Split plan into executable, keep, review, bloc...",execution_manifest_ready_*.parquet; rollback_m...,C:\00_Developement\sch-file-organizer\data\out...,True


,preset_name,preset_goal,stage,notebook,branch,optional,status,latest_file,description,preset_notes,recommended
0,review_only,"Inspect current files, duplicates, junk, extra...",01_policy_check,01_policy_check.ipynb,core,False,done,NaN,Validate YAML policy and vocabularies.,Best default for a new batch or after policy c...,False
1,review_only,"Inspect current files, duplicates, junk, extra...",02_inventory,02_inventory.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,Scan files and write inventory parquet/csv.,Best default for a new batch or after policy c...,False
2,review_only,"Inspect current files, duplicates, junk, extra...",03_rule_classification,03_rule_classification.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,Apply deterministic rules to inventory.,Best default for a new batch or after policy c...,False
3,review_only,"Inspect current files, duplicates, junk, extra...",04_extract_text,04_extract_text.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,Extract text previews from supported file types.,Best default for a new batch or after policy c...,False
4,review_only,"Inspect current files, duplicates, junk, extra...",05_review_outputs,05_review_outputs.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,Merge outputs into one review frame.,Best default for a new batch or after policy c...,False
5,review_only,"Inspect current files, duplicates, junk, extra...",06_planner,06_planner.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,Build conservative dry-run plan.,Best default for a new batch or after policy c...,False
6,review_only,"Inspect current files, duplicates, junk, extra...",07_execution_manifest,07_execution_manifest.ipynb,core,False,done,C:\00_Developement\sch-file-organizer\data\out...,"Split plan into executable, keep, review, bloc...",Best default for a new batch or after policy c...,False


In [5]:
next_notebook = summary_df.iloc[0]['next_notebook'] if not summary_df.empty else None
if pd.isna(next_notebook) or next_notebook is None:
    print('All stages for this preset already have outputs.')
else:
    print('Recommended next notebook for preset', PRESET_NAME + ':', next_notebook)


All stages for this preset already have outputs.


In [6]:
summary_path, status_path, plan_path = export_preset_tables(summary_df, status_df, plan_df, OUTPUT_DIR)
print('Wrote:', summary_path)
print('Wrote:', status_path)
print('Wrote:', plan_path)


Wrote: C:\00_Developement\sch-file-organizer\data\outputs\pipeline_preset_summary_latest.csv
Wrote: C:\00_Developement\sch-file-organizer\data\outputs\pipeline_preset_status_latest.csv
Wrote: C:\00_Developement\sch-file-organizer\data\outputs\pipeline_preset_plan_latest.csv
